加载指定目录1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json以_privacy_policy_links结尾的csv，读取里面apk_name和privacy_policy_url，
访问privacy_policy_url，下载privacy_policy的正文内容为markdown，保存到1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_google_play\{apkname}，也用csv文件保存下载结果

In [ ]:
# {f_position}\{s_position}
f_position = "AA2_second_batch"
s_position = "AA6_sisth_100_batch" # AA6_sisth_100_batch, AA7_seventh_100_batch # AA3_third_100_batch, AA4_forth_100_batch, AA5_fifth_100_batch

In [ ]:
from pathlib import Path
import re
import time
import random
from html import unescape

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# =========================
# paths
# =========================
# IN_DIR = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json")

# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA1_first_328_batch\apk_versions_summary_final_processed.csv     328url--66url(排除掉10个)--46app
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_final_processed.csv 
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_final_processed.csv   91url--15url-- 12app
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_final_processed.csv    74url--19url
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_final_processed.csv   77url--18-- (36) --15app
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_final_processed.csv    74url--14-- 12app

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch\apk_versions_summary_final_processed.csv   80url--18url
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA2_second_100_batch\apk_versions_summary_final_processed.csv  82--20
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA3_third_100_batch\apk_versions_summary_final_processed.csv   80--12
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA4_forth_100_batch\apk_versions_summary_final_processed.csv   72--20
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA5_fifth_100_batch\apk_versions_summary_final_processed.csv   72--20
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA6_sisth_100_batch\apk_versions_summary_final_processed.csv
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_seven_100_batch\apk_versions_summary_final_processed.csv
IN_DIR_PATH = Path(rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_final_processed.csv")  

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\privacy_policy_md
OUT_DIR = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\privacy_policy_md")
OUT_CSV = OUT_DIR / "pp_download_results.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [46]:
# =========================
# requests session
# =========================
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)


# =========================
# helpers
# =========================
def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name)
    name = re.sub(r"\s+", "_", name)
    return name[:200] if name else "unknown_apk"


def html_to_text(html: str) -> str:
    html = re.sub(r"(?is)<script.*?>.*?</script>", " ", html)
    html = re.sub(r"(?is)<style.*?>.*?</style>", " ", html)
    html = re.sub(r"(?i)</p>|<br\s*/?>|</div>|</li>|</tr>|</h[1-6]>", "\n", html)
    html = re.sub(r"(?i)<li[^>]*>", "- ", html)
    text = re.sub(r"(?s)<[^>]+>", " ", html)
    text = unescape(text)
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    return text.strip()


def extract_main_html(html: str) -> str:
    """
    尽量抽正文区域；抽不到就返回原始 html
    """
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")

        # 去掉明显噪音
        for tag in soup(["script", "style", "noscript", "svg", "iframe", "footer", "nav"]):
            tag.decompose()

        # 优先正文容器
        candidates = []

        for selector in [
            "main",
            "article",
            '[role="main"]',
            ".content",
            ".main-content",
            ".post-content",
            ".entry-content",
            ".page-content",
            ".policy",
            ".privacy-policy",
        ]:
            for node in soup.select(selector):
                txt = node.get_text(" ", strip=True)
                if len(txt) > 500:
                    candidates.append((len(txt), str(node)))

        if candidates:
            candidates.sort(reverse=True)
            return candidates[0][1]

        # 回退：找文本最多的 div/section
        blocks = []
        for node in soup.find_all(["div", "section"]):
            txt = node.get_text(" ", strip=True)
            if len(txt) > 1000:
                blocks.append((len(txt), str(node)))

        if blocks:
            blocks.sort(reverse=True)
            return blocks[0][1]

        body = soup.body
        if body:
            return str(body)

        return html

    except Exception:
        return html


def html_to_markdown(html: str) -> str:
    """
    优先 markdownify；失败则退化为纯文本
    """
    main_html = extract_main_html(html)

    try:
        from markdownify import markdownify as md
        md_text = md(main_html, heading_style="ATX")
        md_text = unescape(md_text)
        md_text = re.sub(r"\n{3,}", "\n\n", md_text).strip()
        return md_text
    except Exception:
        return html_to_text(main_html)


def fetch_and_save_markdown(apk_name: str, version: str, url: str):
    out_path = OUT_DIR / f"{apk_name}_{version}.md"

    try:
        resp = session.get(url, timeout=(10, 90), allow_redirects=True)
        status = resp.status_code

        if status != 200:
            return {
                "success": False,
                "http_status": status,
                "final_url": resp.url if hasattr(resp, "url") else "",
                "saved_path": "",
                "content_length": 0,
                "error": f"http_{status}",
                "duplicated": 0
            }

        resp.encoding = resp.apparent_encoding or resp.encoding
        md_text = html_to_markdown(resp.text)

        if not md_text or len(md_text.strip()) < 50:
            return {
                "success": False,
                "http_status": status,
                "final_url": resp.url,
                "saved_path": "",
                "content_length": 0,
                "error": "content_too_short",
                "duplicated": 0
            }

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        return {
            "success": True,
            "http_status": status,
            "final_url": resp.url,
            "saved_path": str(out_path),
            "content_length": len(md_text),
            "error": "",
            "duplicated": 0,
        }

    except Exception as e:
        return {
            "success": False,
            "http_status": "",
            "final_url": "",
            "saved_path": "",
            "content_length": 0,
            "error": f"{type(e).__name__}: {e}",
            "duplicated": 0,
        }

In [47]:
# =========================
# load csv
# =========================
df_in = pd.read_csv(IN_DIR_PATH)
df_in.head(), df_in.shape

(                        apk_name  version                    source  \
 0  com.PoxelStudios.CrossyBrakes       27               google_play   
 1  com.PoxelStudios.CrossyBrakes       28               google_play   
 2   com.QuranReading.quranbangla       25  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla       26  apkitself_waybackmachine   
 4        com.RedLineGames.Game49      114               google_play   
 
                                               pp_url  
 0              https://poxelstudios.com/privacy.html  
 1              https://poxelstudios.com/privacy.html  
 2  https://web.archive.org/web/20240416100627/htt...  
 3  https://web.archive.org/web/20240416100627/htt...  
 4                  https://www.tap-nation.io/policy/  ,
 (100, 4))

In [48]:
# =========================
# run
# =========================
from marshal import version


rows = []
total = len(df_in)

downloaded_keys = set()  # record downloaded (apk_name, privacy_policy_url)

for i, row in df_in.iterrows():
    apk_name = row["apk_name"]
    version = row["version"]
    privacy_policy_url = row["pp_url"]

    # normalize url
    if pd.isna(privacy_policy_url) or str(privacy_policy_url).strip() == "":
        rows.append({
            "apk_name": apk_name,
            "version": version,
            "privacy_policy_url": privacy_policy_url,
            "status": "empty_url",
            "duplicated": 0,
        })
        continue

    privacy_policy_url = str(privacy_policy_url).strip()
    key = (apk_name, privacy_policy_url)

    # skip duplicated url under the same apk
    if key in downloaded_keys:
        print(f"[SKIP DUP] {apk_name} | {version} | {privacy_policy_url}")
        rows.append({
            "apk_name": apk_name,
            "version": version,
            "privacy_policy_url": privacy_policy_url,
            "status": "duplicate_url_same_apk",
            "duplicated": 1,
        })
        continue

    downloaded_keys.add(key)

    result = fetch_and_save_markdown(apk_name, version, privacy_policy_url)

    rows.append({
        "apk_name": apk_name,
        "version": version,
        "privacy_policy_url": privacy_policy_url,
        "apkitself_url": result["final_url"],
        "http_status": result["http_status"],
        "success": result["success"],
        "saved_path": result["saved_path"],
        "content_length": result["content_length"],
        "error": result["error"],
        "duplicated": result["duplicated"],
    })

    if (i + 1) % 20 == 0 or (i + 1) == total:
        print(f"Progress: {i + 1}/{total}")

    time.sleep(random.uniform(0.8, 1.6))


# =========================
# save result csv
# =========================
df_out = pd.DataFrame(rows)
df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("Done.")
print("Output CSV:", OUT_CSV)
print("Downloaded:", int(df_out["success"].sum()), "/", len(df_out))

[SKIP DUP] com.PoxelStudios.CrossyBrakes | 28 | https://poxelstudios.com/privacy.html
[SKIP DUP] com.QuranReading.quranbangla | 26 | https://web.archive.org/web/20240416100627/http://www.quranreading.com/apps/Quran-reading-privacy-policy.html
[SKIP DUP] com.RedLineGames.Game49 | 115 | https://www.tap-nation.io/policy/
[SKIP DUP] com.polyverse.bricks.game | 123 | https://www.riveroll.top/privacy-policy-2/
[SKIP DUP] com.portablepixels.smokefree | 6430645 | https://web.archive.org/web/20241005092342/https://smokefreeapp.com/privacy-policy/
[SKIP DUP] com.ppa.real.shooting.strike | 22 | https://web.archive.org/web/20231204001048/https://playpointaus.blogspot.com/2019/02/privacy-policy.html
[SKIP DUP] com.prankphone.broken.screen.diamond.bg | 50 | https://web.archive.org/web/20240927025649/https://bralyvn.com/privacy-policy.php
[SKIP DUP] com.punjabimatrimony | 341 | https://www.punjabimatrimony.com/privacy-policy.php
[SKIP DUP] com.qidafcl.sl054 | 2208 | https://www.crazylabs.com/apps-pri